### Target: next day collisions per nbhd

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "features_nbhd_day.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["nbhd_id","date"])

# next-day collisions target
df["y_next_collisions"] = df.groupby("nbhd_id")["collisions"].shift(-1)

# drop last day per nbhd
df = df.dropna(subset=["y_next_collisions"]).copy()
df["y_next_collisions"] = df["y_next_collisions"].astype(int)
print(df.shape)

(691250, 47)


### Train regressor and evaluate Precision@K

In [2]:
import numpy as np
from lightgbm import LGBMRegressor

split_date = pd.Timestamp("2024-01-01")
train = df[df["date"] < split_date].copy()
test  = df[df["date"] >= split_date].copy()

drop_cols = {"date","area_id","area_name","AREA_ID","AREA_NAME","geometry","y_next_collisions","NEIGHBOURHOOD_158","NEIGHBOURHOOD_140"}
X_cols = [c for c in train.columns if c not in drop_cols and c not in ["collisions"]]

reg = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
reg.fit(train[X_cols], train["y_next_collisions"])

test["risk_score"] = reg.predict(test[X_cols])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1826
[LightGBM] [Info] Number of data points in the train set: 575910, number of used features: 40
[LightGBM] [Info] Start training from score 0.942746


### Precision@K evaluation, top 10 neighbourhoods each day

In [3]:
def precision_at_k(day_df, k=10):
    # true top-k by actual next collisions
    true_top = set(day_df.sort_values("y_next_collisions", ascending=False).head(k)["nbhd_id"])
    pred_top = set(day_df.sort_values("risk_score", ascending=False).head(k)["nbhd_id"])
    return len(true_top & pred_top) / k

daily = test.groupby("date").apply(lambda g: precision_at_k(g, k=10))
print("Mean Precision@10:", daily.mean())

Mean Precision@10: 0.38109589041095887


### Save ranking predictions

In [4]:
out_path = PROCESSED_DIR / "pred_nbhd_risk_ranking.parquet"
test[["date","nbhd_id","risk_score","y_next_collisions"]].to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\processed\pred_nbhd_risk_ranking.parquet
